# Azure AI Foundry Agent Service - MCP ツール実行

Azure AI Foundry Agent Service で MCP ツールを利用する場合、すでに稼働している MCP サーバーエンドポイントを用意する必要があります。

このノートブックでは、MCP ツールを活用して Microsoft Learn 公式ドキュメントを検索できるエージェントを作成します。
MCP プロトコルにより、エージェントが外部の API やサービスと安全かつ標準化された方法で連携する方法を学習します。

- 参考：[Microsoft Learn MCP Server overview](https://learn.microsoft.com/en-us/training/support/mcp)

## 事前準備

### 依存関係のインストール

In [1]:
%pip install -r ../requirements.txt --quiet


[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### 必要なモジュールのインポート

In [2]:
import os
import time
import json

from dotenv import load_dotenv
from IPython.display import Image, display

from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.agents.models import (
    ListSortOrder,
    McpTool,
    RunStepToolCallDetails,
    RunStepFunctionToolCall,
    RunStepMessageCreationDetails,
    RequiredMcpToolCall,
    SubmitToolApprovalAction,
    ToolApproval,
)

### 環境変数の読み込み

In [3]:
load_dotenv(override=True)

PROJECT_ENDPOINT=os.getenv("PROJECT_ENDPOINT")
AZURE_DEPLOYMENT_NAME=os.getenv("AZURE_DEPLOYMENT_NAME")

### クライアントの初期化

In [4]:
# AI Project Client を初期化
project_client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential()
)

# AgentClient の作成
agents_client = project_client.agents

### ユーティリティ関数

エージェントの実行結果を表示するためのヘルパー関数を定義します。

`agent_run_outputs`関数は以下の機能を提供します：
- スレッド内のメッセージ一覧を取得・表示
- 画像コンテンツがある場合は保存・表示
- ツール呼び出し情報の詳細表示（Run Stepsから取得）

In [5]:
def agent_run_outputs(thread_id, agents_client, target_dir="./output_images", show_tool_calls=True, run_id=None):
    """
    指定したスレッドIDのRun実行結果（テキスト・画像・ツール呼び出し）をNotebook上に表示＆画像は保存。
    
    Args:
        thread_id: スレッドID
        agents_client: エージェントクライアント
        target_dir: 画像保存ディレクトリ
        show_tool_calls: ツール呼び出し情報を表示するかどうか
        run_id: 特定のRunのツール呼び出し情報を表示する場合のRun ID
    """    
    messages = agents_client.messages.list(thread_id=thread_id, order=ListSortOrder.ASCENDING)
    os.makedirs(target_dir, exist_ok=True)

    # メッセージの重複防止
    displayed_message_ids = set()
    
    # メッセージの表示
    for message in messages:
        # メッセージの重複チェック
        if message.id in displayed_message_ids:
            continue
        displayed_message_ids.add(message.id)
        
        print(f"\n{'='*60}")
        print(f"MESSAGE ROLE: {message.role.upper()}")
        print(f"MESSAGE ID: {message.id}")
        print(f"{'='*60}")
        
        # テキスト出力
        if message.text_messages:
            for txt in message.text_messages:
                print(f"{txt.text.value}")
        
        # 画像出力
        if hasattr(message, "image_contents") and message.image_contents:
            print(f"\n[IMAGES]")
            for image_content in message.image_contents:
                file_id = image_content.image_file.file_id
                file_name = f"{file_id}_image_file.png"

                agents_client.files.save(
                    file_id=file_id,
                    file_name=file_name,
                    target_dir=target_dir
                )
                print(f"  Saved image: {file_name}")
                display(Image(filename=f"{target_dir}/{file_name}"))
    
    # ツール呼び出し情報の表示（Run Stepsから取得）
    if show_tool_calls and run_id:
        print(f"\n{'='*60}")
        print(f"RUN STEPS INFORMATION (RUN ID: {run_id})")
        print(f"{'='*60}")
        
        try:
            # Run Stepsを取得（デフォルトは新しい順なので、古い順に並び替え）
            run_steps = agents_client.run_steps.list(thread_id=thread_id, run_id=run_id)
            run_steps_list = list(run_steps)
            run_steps_list.reverse()  # 実行順序に並び替え（STEP1から順番に）
            
            # 重複防止のためのセット
            displayed_step_ids = set()
            
            print(f"Total Run Steps: {len(run_steps_list)}")
            
            # 全てのrun stepsを実行順序で表示
            for step_num, run_step in enumerate(run_steps_list, 1):
                # 重複チェック
                if run_step.id in displayed_step_ids:
                    print(f"[STEP {step_num}] - SKIPPED (Duplicate Step ID: {run_step.id})")
                    continue
                displayed_step_ids.add(run_step.id)
                
                print(f"\n[STEP {step_num}] - {run_step.type}")
                print(f"  Step ID: {run_step.id}")
                print(f"  Status: {run_step.status}")
                
                # Message Creation Step
                if isinstance(run_step.step_details, RunStepMessageCreationDetails):
                    print(f"  Message Creation Step")
                    if hasattr(run_step.step_details.message_creation, 'message_id'):
                        print(f"  Message ID: {run_step.step_details.message_creation.message_id}")
                
                # Tool Calls Step
                elif isinstance(run_step.step_details, RunStepToolCallDetails):
                    print(f"  Tool Calls Step - {len(run_step.step_details.tool_calls)} tool(s)")
                    
                    for tool_num, tool_call in enumerate(run_step.step_details.tool_calls, 1):
                        print(f"\n    [TOOL CALL {tool_num}]")
                        print(f"    Tool Type: {tool_call.type}")
                        print(f"    Tool Call ID: {tool_call.id}")
                        
                        # Function Tool Call の詳細
                        if isinstance(tool_call, RunStepFunctionToolCall):
                            print(f"    Function Name: {tool_call.function.name}")
                            print(f"    Function Arguments: {tool_call.function.arguments}")
                            # 関数の実行結果を表示（利用可能な場合）
                            if hasattr(tool_call.function, 'output') and tool_call.function.output:
                                print(f"    Function Output: {tool_call.function.output}")
                            elif hasattr(tool_call.function, 'outputs') and tool_call.function.outputs:
                                print(f"    Function Outputs: {tool_call.function.outputs}")
                            elif hasattr(tool_call.function, 'result') and tool_call.function.result:
                                print(f"    Function Result: {tool_call.function.result}")
                        
                        print(f"    {'-'*30}")
                
                # その他のステップタイプ
                else:
                    print(f"  Step Type: {type(run_step.step_details).__name__}")
                
                print(f"  Created At: {run_step.created_at}")
                if hasattr(run_step, 'completed_at') and run_step.completed_at:
                    print(f"  Completed At: {run_step.completed_at}")
                
                print(f"  {'='*50}")
                
        except Exception as e:
            print(f"Error retrieving run steps: {e}")
            print(f"Run ID: {run_id}, Thread ID: {thread_id}")

## Azure AI Agent Service

### McpTool の定義

**MCP ツール設定項目**
- **server_label**: MCPサーバーの一意識別子
- **server_url**: APIのエンドポイント
- **allowed_tools**: 利用可能なツールのリスト（オプション）

In [6]:
mcp_tool = McpTool(
    server_label="MicrosoftDocs",
    server_url="https://learn.microsoft.com/api/mcp",
    allowed_tools=["microsoft_docs_search"] # Option
)

### エージェントの作成

`create_agent` メソッドを用いてエージェント作成します。

In [7]:
mcp_agent = agents_client.create_agent(
    model=AZURE_DEPLOYMENT_NAME,
    name="mcp_agent",
    instructions=(
        "あなたは、MCPツールを使用してユーザーを支援できる有用なエージェントです。"
        "利用可能なMCPツールを使用して、質問に答えてタスクを実行します。"    
    ),
    tools=mcp_tool.definitions,
    tool_resources=mcp_tool.resources,
)
print(f"Created Agent. AGENT_ID: {mcp_agent.id}")


Created Agent. AGENT_ID: asst_Yd4qtVEbO9GO2SewL29CR6nd


In [8]:
agent_dict = mcp_agent.as_dict()
print(json.dumps(agent_dict, indent=2, ensure_ascii=False))

{
  "id": "asst_Yd4qtVEbO9GO2SewL29CR6nd",
  "object": "assistant",
  "created_at": 1758891431,
  "name": "mcp_agent",
  "description": null,
  "model": "gpt-4.1-mini",
  "instructions": "あなたは、MCPツールを使用してユーザーを支援できる有用なエージェントです。利用可能なMCPツールを使用して、質問に答えてタスクを実行します。",
  "tools": [
    {
      "type": "mcp",
      "server_label": "MicrosoftDocs",
      "server_url": "https://learn.microsoft.com/api/mcp",
      "allowed_tools": [
        "microsoft_docs_search"
      ]
    }
  ],
  "top_p": 1.0,
  "temperature": 1.0,
  "tool_resources": {},
  "metadata": {},
  "response_format": "auto"
}


### スレッドの初期化

スレッドを初期化し、ユーザーメッセージをスレッドに登録します。

In [9]:
# Thread の作成
thread = agents_client.threads.create()
print(f"Created Thread. THREAD_ID: {thread.id}")

Created Thread. THREAD_ID: thread_o1t8qM10HimOpLGATu52GNxD


### メッセージを追加

スレッドに、ユーザーメッセージを書き込みます。

In [10]:
# メッセージの追加
user_message = "Azure AI Foundry Agent Service の最新情報を教えてください。"

message = agents_client.messages.create(
    thread_id=thread.id,
    role="user",
    content=user_message,
)

print(f"Added Message. MESSAGE_ID: {message.id}")

Added Message. MESSAGE_ID: msg_ec7LRi2Vc7kcyjIldGl9emQc


### Run の実行

作成したエージェントとスレッドを指定して `Run` を実行し、エージェントから回答を生成します。

**MCP ツールを自動承認して実行**

通常、MCP ツールの実行にはユーザーの承認が必要です。
ここでは **承認を自動的に許可する設定**（`mcp_tool.set_approval_mode = "never"`）を用いてエージェントを実行します。

承認モード（`mcp_tool.set_approval_mode`）には次の2種類があります。

1. **always** – すべてのツール呼び出しで承認が必要 ※ デフォルト
2. **never** – すべてのツール呼び出しが承認不要（自動承認）


In [11]:
# MCP ツール実行時の承認モードを never に設定
mcp_tool.set_approval_mode("never")

print(mcp_tool.resources)

{'mcp': [{'server_label': 'MicrosoftDocs', 'headers': {}, 'require_approval': 'never'}]}


In [ ]:
run = agents_client.runs.create_and_process(
    thread_id=thread.id,
    agent_id=mcp_agent.id,
    tool_resources=mcp_tool.resources # MCP ツールのリソース情報を渡す（承認モード設定やリクエストヘッダーの設定はここで定義される）
)

if run.status == "failed":
    print(f"Run failed: {run.last_error}")
else:
    agent_run_outputs(thread.id, agents_client, show_tool_calls=True, run_id=run.id)


MESSAGE ROLE: USER
MESSAGE ID: msg_ec7LRi2Vc7kcyjIldGl9emQc
Azure AI Foundry Agent Service の最新情報を教えてください。

MESSAGE ROLE: ASSISTANT
MESSAGE ID: msg_Ch83ieHzRTSoukIbSKV0cZy5
Azure AI Foundry Agent Service の最新情報についてまとめます。

2025年5月の最新GAリリースでは、以下の主な新機能が追加されています。
- Azure AI Foundry Visual Studio Code 拡張機能が利用可能に。エージェントのデプロイや設定がVS Code内で可能
- Connected agents 機能で複数の目的特化エージェントの連携が容易に。外部オーケストレータ不要
- Trace agents 機能でエージェントの実行フローを追跡し入力や出力を詳細にデバッグ可能
- Azure Logic Apps トリガー連携でイベント発生でエージェントを自動起動
- 新しいエージェントツールとして Bing Custom Search、Morningstar 追加

2025年8月の更新には以下があります。
- Java SDKのパブリックプレビュー開始。コードサンプル多数公開
- Browser Automation ツールがパブリックプレビューで利用可能に。Microsoft Playwright Workspaces と連携しブラウザ操作を自然言語指示で実行できる
- 新リージョン対応 : ブラジル南部、ドイツ西部中央、イタリア北部、米国南中部

その他の期間の主な更新例としては、
- 2025年3月にMicrosoft Fabric ツール追加でFabric上のデータにチャットでアクセス可能
- 2025年2月にAzure AI Foundry ポータルでコード不要のエージェント作成・デバッグ対応
- 2024年12月にAzure AI Service プレビュー公開。Azure OpenAIモデルに加えLlamaなどの他社モデルサポートとエンタープライズ向け機能

詳細や追加のアップデートは公式ドキュメント
https://learn.microsoft.com

### Option

**MCP ツール手動承認で実行**

エージェントが MCP ツールの実行が必要と判断した際、クライアント側にて手動の承認を行うプロセスを実装します。主に、ユーザーインターフェース上で「○○ツールを使っていいですか？」という確認プロセスを実装したい時にこのような構成をとります。

手動承認の実装流れは以下になります。
1. `create()`で Run を実行
2. MCP ツール実行が必要と判断されたら、Run は待機状態へ
3. クライアント側から `submit_tool_outputs()`で承認結果を送信
4. Agent Service 上の Run が再開（MCP ツールを実行して回答を生成）

**MCP ツールの設定更新**

In [13]:
# MCP ツール実行時の承認モードを always に設定更新
mcp_tool.set_approval_mode("always")

print(mcp_tool.resources)

{'mcp': [{'server_label': 'MicrosoftDocs', 'headers': {}, 'require_approval': 'always'}]}


**後続処理をまとめて実行**

In [14]:
# Thread の作成
thread_1 = agents_client.threads.create()
print(f"Created Thread. THREAD_ID: {thread_1.id}")

# メッセージの追加
user_message = "Azure AI Foundry Agent Service の最新情報を教えてください。"
message = agents_client.messages.create(
    thread_id=thread_1.id,
    role="user",
    content=user_message,
)
print(f"Added Message. MESSAGE_ID: {message.id}")

# Run の作成と実行
run_1 = agents_client.runs.create(
    thread_id=thread_1.id,
    agent_id=mcp_agent.id,
    tool_resources=mcp_tool.resources # 承認モード always で実行
    
)

# Run の状態をポーリングして、必要なアクションがあるか確認
while True:
    time.sleep(1)
    run = agents_client.runs.get(thread_id=thread_1.id, run_id=run_1.id)

    if run.status == "requires_action" and isinstance(run.required_action, SubmitToolApprovalAction):
        tool_calls = run.required_action.submit_tool_approval.tool_calls
        if not tool_calls:
            print("No tool calls provided - cancelling run")
            agents_client.runs.cancel(thread_id=thread_1.id, run_id=run_1.id)
            break

        tool_approvals = []
        for tool_call in tool_calls:
            if isinstance(tool_call, RequiredMcpToolCall):
                try:
                    print(f"Approving tool call: {tool_call}")
                    tool_approvals.append(
                        ToolApproval(
                            tool_call_id=tool_call.id,
                            approve=True,  # 承認する場合は True、拒否する場合は False
                            headers=mcp_tool.headers,
                        )
                    )
                except Exception as e:
                    print(f"Error approving tool_call {tool_call.id}: {e}")

        print(f"tool_approvals: {tool_approvals}")
        if tool_approvals:
            # ツール承認を送信
            agents_client.runs.submit_tool_outputs(
                thread_id=thread_1.id,
                run_id=run_1.id,
                tool_approvals=tool_approvals
            )

    print(f"Current run status: {run.status}")
    if run.status not in ["queued", "in_progress", "requires_action"]:
        break


Created Thread. THREAD_ID: thread_CZRzuQQX3IFz5fTLzUHW8Mrg
Added Message. MESSAGE_ID: msg_G1TKEF8748yS43iGTKg9dxGv
Current run status: RunStatus.IN_PROGRESS
Approving tool call: {'id': 'call_NdLsud4ASSXDr9Tw07EBqquq', 'type': 'mcp', 'arguments': '{"query":"Azure AI Foundry Agent Service latest updates","question":"Azure AI Foundry Agent Service ã\x81®æ\x9c\x80æ\x96°æ\x83\x85å\xa0±ã\x82\x92æ\x95\x99ã\x81\x88ã\x81¦ã\x81\x8fã\x81\xa0ã\x81\x95ã\x81\x84ã\x80\x82"}', 'name': 'microsoft_docs_search', 'server_label': 'MicrosoftDocs'}
tool_approvals: [{'tool_call_id': 'call_NdLsud4ASSXDr9Tw07EBqquq', 'approve': True, 'headers': {}}]
Current run status: RunStatus.REQUIRES_ACTION
Current run status: RunStatus.IN_PROGRESS
Current run status: RunStatus.IN_PROGRESS
Current run status: RunStatus.IN_PROGRESS
Current run status: RunStatus.IN_PROGRESS
Current run status: RunStatus.COMPLETED


In [15]:
agent_run_outputs(thread_1.id, agents_client, show_tool_calls=True, run_id=run_1.id)


MESSAGE ROLE: USER
MESSAGE ID: msg_G1TKEF8748yS43iGTKg9dxGv
Azure AI Foundry Agent Service の最新情報を教えてください。

MESSAGE ROLE: ASSISTANT
MESSAGE ID: msg_M6PFcbCQcOxfTMDp00T66eZs
Azure AI Foundry Agent Service の最新情報は以下のとおりです（2025年8月時点）：

1. Java SDK がパブリックプレビューで利用可能になりました。Java向けのクイックスタートや各種ツールのサンプルコードも公開されています。
2. Browser Automation（ブラウザ自動化）ツールがパブリックプレビューで利用可能になりました。このツールはMicrosoft Playwright Workspacesを用いて、自然言語プロンプトでブラウザ作業を自動化できます。
3. 新しいリージョンで利用可能になりました。ブラジル南部、ドイツ西中央、イタリア北部、南中央米国リージョンです。
4. 6月にはDeep Researchツール（複数ステップの調査プロセスを実現）やModel Context Protocol（MCP）ツール（リモートのMCPサーバーに接続できるツール）が公開されています。

詳細は公式ドキュメントをご覧ください：
https://learn.microsoft.com/en-us/azure/ai-foundry/agents/whats-new

他にもGA（一般提供）化されたり、新しいエージェントツールやVisual Studio Code拡張機能、Azure Logic Appsからエージェントをトリガーできる機能などが追加されています。ご興味あれば特定の機能についても詳しくご案内可能です。

RUN STEPS INFORMATION (RUN ID: run_mdZE8uAlHjY9hSlTVG6bu0PM)
Total Run Steps: 2

[STEP 1] - RunStepType.TOOL_CALLS
  Step ID: step_phuOfqdoUuJ5kV2I6k1AhOuP
  Status: RunStepStatus.COMPLET

### エージェントとスレッドの削除

最後に、当ノートブックで作成したエージェントとスレッドを削除します。

In [16]:
# Agent の削除
agents_client.delete_agent(agent_id=mcp_agent.id)
print("Agent を削除しました。")

Agent を削除しました。
